In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In the **Data** tab, search for and add the dataset:  
**"Protein Secondary Structure (CASP12, CB513, TS115)"** by `tamzidhasan`.

This dataset contains three columns:
- **`seq`** : primary amino acid sequence (string)
- **`sst3`** : secondary structure in **3 states**  
  - `H` = helix (includes G, H, I)  
  - `E` = sheet (includes B, E)  
  - `C` = coil (includes T, S, C)
- **`sst8`** : secondary structure in **8 states**  
  - `H` = α-helix, `B` = β-bridge, `E` = β-strand, `G` = 3₁₀-helix,  
    `I` = π-helix, `T` = turn, `S` = bend, `C` = coil/loop

For this tutorial we will use **`sst3`** (3‑class prediction) because it is simpler. The `sst8` column can be used for more detailed prediction if desired.



In [ ]:
# import libraries
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
# Install ESM (if not already installed)
!pip install fair-esm

# Import the ESM library
import esm

In [ ]:
!ls /kaggle/input/datasets/tamzidhasan/protein-secondary-structure-casp12-cb513-ts115

In [ ]:
train_df = pd.read_csv('/kaggle/input/datasets/tamzidhasan/protein-secondary-structure-casp12-cb513-ts115/training_secondary_structure_train.csv')
valid_df = pd.read_csv('/kaggle/input/datasets/tamzidhasan/protein-secondary-structure-casp12-cb513-ts115/validation_secondary_structure_valid.csv')

test_df = pd.read_csv('/kaggle/input/datasets/tamzidhasan/protein-secondary-structure-casp12-cb513-ts115/test_secondary_structure_casp12.csv')

ss3_map = {'H':0, 'E':1, 'C':2}
window_size = 15

print(f"Training proteins: {len(train_df)}")
print(f"Validation proteins: {len(valid_df)}")
print(f"Test (CASP12) proteins: {len(test_df)}")
print("Columns:", train_df.columns.tolist())

In [ ]:
# ------------------------------------------------------------
# 2. Preprocessing functions
# ------------------------------------------------------------
def one_hot_encode_window(window_seq, aa_to_idx):
    window_size = len(window_seq)
    one_hot = np.zeros((window_size, 20))
    for j, aa in enumerate(window_seq):
        if aa in aa_to_idx:
            one_hot[j, aa_to_idx[aa]] = 1
    return one_hot.flatten()

def prepare_data_onehot(sequences, structures, window_size=15):
    pad_len = window_size // 2
    aa_order = 'ACDEFGHIKLMNPQRSTVWY'
    aa_to_idx = {aa:i for i,aa in enumerate(aa_order)}
    X_list, y_list = [], []
    for seq, ss in zip(sequences, structures):
        padded_seq = 'X'*pad_len + seq + 'X'*pad_len
        padded_ss  = 'C'*pad_len + ss + 'C'*pad_len
        for i in range(len(seq)):
            window_seq = padded_seq[i:i+window_size]
            X_list.append(one_hot_encode_window(window_seq, aa_to_idx))
            y_list.append(ss3_map[padded_ss[i+pad_len]])
    return np.array(X_list), np.array(y_list)


def prepare_data_esm(sequences, structures, esm_model, batch_converter, device='cuda', window_size=15):
    pad_len = window_size // 2
    protein_list = [(f'prot_{i}', seq) for i, seq in enumerate(sequences)]
    batch_labels, batch_strs, batch_tokens = batch_converter(protein_list)
    batch_tokens = batch_tokens.to(device)
    with torch.no_grad():
        results = esm_model(batch_tokens, repr_layers=[33])
        embeddings = results["representations"][33]  # (n_proteins, max_len, 1280)
        embeddings = embeddings.cpu().numpy()
    
    X_list, y_list = [], []
    for idx, (seq, ss) in enumerate(zip(sequences, structures)):
        prot_emb = embeddings[idx]  # (L, 1280)
        pad_vec = np.zeros((1, 1280))
        padded_emb = np.vstack([pad_vec]*pad_len + [prot_emb] + [pad_vec]*pad_len)
        padded_ss = 'C'*pad_len + ss + 'C'*pad_len
        for i in range(len(seq)):
            window_emb = padded_emb[i:i+window_size]  # (window_size, 1280)
            X_list.append(window_emb.flatten())
            y_list.append(ss3_map[padded_ss[i+pad_len]])
    return np.array(X_list), np.array(y_list)


def prepare_data_esm_chunked(sequences, structures, esm_model, batch_converter, 
                             device, window_size=15, chunk_size=100):
    """
    Extract ESM embeddings for many proteins in chunks to avoid memory overflow.
    """
    pad_len = window_size // 2
    all_X, all_y = [], []
    n_proteins = len(sequences)
    
    for start in range(0, n_proteins, chunk_size):
        end = min(start + chunk_size, n_proteins)
        print(f"Processing proteins {start} to {end-1}...")
        
        # Build chunk of proteins
        chunk_seqs = sequences[start:end]
        chunk_ss = structures[start:end]
        protein_list = [(f'prot_{i}', seq) for i, seq in enumerate(chunk_seqs)]
        
        # Convert to tokens and get embeddings
        batch_labels, batch_strs, batch_tokens = batch_converter(protein_list)
        batch_tokens = batch_tokens.to(device)
        with torch.no_grad():
            results = esm_model(batch_tokens, repr_layers=[33])
            if 33 in results["representations"]:
                embeddings = results["representations"][33].cpu().numpy()
            else:
                last_key = max(results["representations"].keys())
                print(f"Layer 33 not found, using layer {last_key} instead.")
                embeddings = results["representations"][last_key].cpu().numpy()
        
        # Slide window for each protein in chunk
        for idx, (seq, ss) in enumerate(zip(chunk_seqs, chunk_ss)):
            prot_emb = embeddings[idx][:len(seq)]  # trim to actual length
            # Pad with zero vectors
            pad_vec = np.zeros((1, 1280))
            padded_emb = np.vstack([pad_vec]*pad_len + [prot_emb] + [pad_vec]*pad_len)
            padded_ss = 'C'*pad_len + ss + 'C'*pad_len
            for i in range(len(seq)):
                window_emb = padded_emb[i:i+window_size]
                all_X.append(window_emb.flatten())
                all_y.append(ss3_map[padded_ss[i+pad_len]])
    
    return np.array(all_X), np.array(all_y)


In [ ]:
# ------------------------------------------------------------
# 3. MLP Model
# ------------------------------------------------------------
class SimpleMLP(nn.Module):
    def __init__(self, input_dim, hidden_dim=256, output_dim=3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(hidden_dim, hidden_dim//2),
            nn.ReLU(),
            nn.Linear(hidden_dim//2, output_dim)
        )
    def forward(self, x):
        return self.net(x)


def get_default_device():
    """Return 'cuda' only if a compatible GPU is available."""
    if torch.cuda.is_available():
        # Check compute capability
        cap = torch.cuda.get_device_capability()
        if cap[0] >= 7:   # Requires at least sm_70
            return torch.device('cuda')
        else:
            print(f"GPU compute capability {cap[0]}.{cap[1]} is < 7.0. Falling back to CPU.")
    return torch.device('cpu')

# In train_and_evaluate, replace device detection with:
device = get_default_device()

# ------------------------------------------------------------
# 4. Training function with GPU support
# ------------------------------------------------------------
def train_and_evaluate(X_train, y_train, X_valid, y_valid, X_test, y_test,
                       input_dim, epochs=20, batch_size=64, lr=0.001, device=None):
    if device is None:
        device = get_default_device()
    print(f"Using device: {device}")
    
    model = SimpleMLP(input_dim).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    
    # Convert to tensors and move to device
    X_train_t = torch.tensor(X_train, dtype=torch.float32).to(device)
    y_train_t = torch.tensor(y_train, dtype=torch.long).to(device)
    X_valid_t = torch.tensor(X_valid, dtype=torch.float32).to(device)
    y_valid_t = torch.tensor(y_valid, dtype=torch.long).to(device)
    
    train_dataset = TensorDataset(X_train_t, y_train_t)
    valid_dataset = TensorDataset(X_valid_t, y_valid_t)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    valid_loader = DataLoader(valid_dataset, batch_size=batch_size, shuffle=False)
    
    best_val_acc = 0
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        for batch_x, batch_y in train_loader:
            optimizer.zero_grad()
            outputs = model(batch_x)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        
        # Validation
        model.eval()
        correct = 0
        total = 0
        with torch.no_grad():
            for batch_x, batch_y in valid_loader:
                outputs = model(batch_x)
                _, preds = torch.max(outputs, 1)
                correct += (preds == batch_y).sum().item()
                total += batch_y.size(0)
        val_acc = correct / total
        if val_acc > best_val_acc:
            best_val_acc = val_acc
        if (epoch+1) % 5 == 0:
            print(f"Epoch {epoch+1:2d} | Loss: {total_loss/len(train_loader):.4f} | Val Acc: {val_acc:.4f}")
    
    # Test evaluation
    X_test_t = torch.tensor(X_test, dtype=torch.float32).to(device)
    y_test_t = torch.tensor(y_test, dtype=torch.long).to(device)
    model.eval()
    with torch.no_grad():
        outputs = model(X_test_t)
        _, preds = torch.max(outputs, 1)
        test_acc = (preds == y_test_t).float().mean().item()
    print(f"Test Accuracy: {test_acc:.4f}")
    return test_acc, best_val_acc

# ------------------------------------------------------------
# 5. Run experiments (GPU automatically used if available)
# ------------------------------------------------------------
train_seqs = train_df['seq'].tolist()
train_ss3 = train_df['sst3'].tolist()
valid_seqs = valid_df['seq'].tolist()
valid_ss3 = valid_df['sst3'].tolist()
test_seqs = test_df['seq'].tolist()
test_ss3 = test_df['sst3'].tolist()

# ---- One-hot ----
print("=== One-hot encoding ===")
X_train_oh, y_train_oh = prepare_data_onehot(train_seqs, train_ss3, window_size)
X_valid_oh, y_valid_oh = prepare_data_onehot(valid_seqs, valid_ss3, window_size)
X_test_oh, y_test_oh   = prepare_data_onehot(test_seqs, test_ss3, window_size)
input_dim_oh = X_train_oh.shape[1]
acc_oh, val_oh = train_and_evaluate(X_train_oh, y_train_oh, X_valid_oh, y_valid_oh,
                                    X_test_oh, y_test_oh, input_dim_oh, epochs=15)

# ---- ESM ----
print("\n=== ESM embeddings ===")
subset_size = 200   # change to None to use all
if subset_size:
    train_seqs = train_df['seq'].tolist()[:subset_size]
    train_ss3 = train_df['sst3'].tolist()[:subset_size]
    valid_seqs = valid_df['seq'].tolist()[:subset_size]
    valid_ss3 = valid_df['sst3'].tolist()[:subset_size]
    test_seqs = test_df['seq'].tolist()[:subset_size]
    test_ss3 = test_df['sst3'].tolist()[:subset_size]
else:
    train_seqs = train_df['seq'].tolist()
    train_ss3 = train_df['sst3'].tolist()
    valid_seqs = valid_df['seq'].tolist()
    valid_ss3 = valid_df['sst3'].tolist()
    test_seqs = test_df['seq'].tolist()
    test_ss3 = test_df['sst3'].tolist()
    
device = get_default_device()          # respects capability
print(f"Using device for ESM: {device}")

esm_model, alphabet = esm.pretrained.esm2_t12_35M_UR50D()   #esm2_t12_35M_UR50D
esm_model = esm_model.to(device)
batch_converter = alphabet.get_batch_converter()


    

# Extract embeddings (this may take a few minutes)
# Prepare datasets
X_train_esm, y_train_esm = prepare_data_esm_chunked(
    train_seqs, train_ss3, esm_model, batch_converter, device, window_size, chunk_size=50)
X_valid_esm, y_valid_esm = prepare_data_esm_chunked(
    valid_seqs, valid_ss3, esm_model, batch_converter, device, window_size, chunk_size=50)
X_test_esm, y_test_esm = prepare_data_esm_chunked(
    test_seqs, test_ss3, esm_model, batch_converter, device, window_size, chunk_size=50)

# X_train_esm, y_train_esm = prepare_data_esm(train_seqs, train_ss3, esm_model, batch_converter, device, window_size)
# X_valid_esm, y_valid_esm = prepare_data_esm(valid_seqs, valid_ss3, esm_model, batch_converter, device, window_size)
# X_test_esm, y_test_esm   = prepare_data_esm(test_seqs, test_ss3, esm_model, batch_converter, device, window_size)
input_dim_esm = X_train_esm.shape[1]
acc_esm, val_esm = train_and_evaluate(X_train_esm, y_train_esm, X_valid_esm, y_valid_esm,
                                      X_test_esm, y_test_esm, input_dim_esm, epochs=15)

# ---- Comparison ----
print("\n=== Final Comparison ===")
print(f"One-hot encoding: Best Val = {val_oh:.4f}, Test = {acc_oh:.4f}")
print(f"ESM embeddings:  Best Val = {val_esm:.4f}, Test = {acc_esm:.4f}")